# HELMETY8 v5 — EdgeVision training from Google Drive
**Updated from your `HELMETY8_v4.ipynb`: same YOLOv8m main model, two classes, local dataset files.**

This notebook replaces the original Roboflow download cell with loading from your own Drive folder. It adds dataset checks, cleaning, two-class conversion, grouped train/validation/test splitting, training-only balancing, explicit augmentation settings, persistent checkpoints, and evaluation of no-helmet detection.

The app's class IDs/names stay **0 = `accept-Helmet-`**, **1 = `non-Helmet-`**. The 13-class backup model and your existing app are not edited here. This retrains YOLOv8m from the same official pretrained starting model used by your original notebook; it does not require uploading your old weights.

## Prepare your folder once — no code edits needed
1. In **Google Drive → My Drive**, create a folder named **`Helmet_Project`**.
2. Put **this notebook and the FULL EdgeVision dataset ZIP** in that folder. Keep its original ZIP filename. The full ZIP must contain the 2,392 images and `labels/yolo` files. The 209-image sample alone is insufficient.
3. Open this notebook with Google Colab. Select **Runtime → Change runtime type → T4 GPU**.
4. Run the numbered code cells from top to bottom. Connect your Drive when Google asks. Text paragraphs are explanations, not code.

**The fixed shared folder is `/content/drive/MyDrive/Helmet_Project`.** Colab cannot directly read a folder on your laptop. Source images are copied from Drive to Colab's local disk for faster training. Checkpoints and reports are saved back to the same project folder, under `outputs/<run name>/`.

An extracted EdgeVision folder also works. Separate image and label ZIPs in the project folder work only if, together, they contain the complete dataset. Keep only this dataset version's files there.

## What has and has not been checked
All 2,392 YOLO label files and the 209 images you supplied were examined. That sample is much more relevant to motorcycle riders than your earlier mixed dataset. It also contains annotation errors and repeated scenes. The code excludes four confirmed problematic examples and applies structural checks to every available image/label.

**Automatic cleaning cannot discover every missing head or wrong semantic label.** The other images have not been visually reviewed here. Training, achieved accuracy, Drive authorization and a full Colab run have not been executed here; preprocessing was checked on your supplied sample. A result above 75% is a target, not a guarantee.


## 1 — Install the required libraries
Ultralytics provides YOLOv8. Pillow reads images; SciPy/NumPy support similar-image grouping; YAML/Pandas handle configuration and reports. No Roboflow account or API key is needed for this file-based workflow. The official pretrained weights download automatically later.


In [ ]:
%pip -q install "ultralytics==8.3.162" scipy pandas pyyaml pillow


## 2 — Connect Google Drive and select the shared project folder
Place the dataset ZIP beside this notebook in `My Drive/Helmet_Project` before running this cell. The folder name is deliberate: it avoids needing to edit paths.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Helmet_Project')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError('Create My Drive/Helmet_Project, put the notebook and full dataset ZIP there, then rerun this cell.')
print('Project folder:', PROJECT_DIR)
print('ZIP files:', [p.name for p in PROJECT_DIR.glob('*.zip')])


## 3 — Check the GPU and create or resume a run
On the first run, this creates a new output folder. If saved checkpoints already exist, enter their displayed number to continue that run, or press Enter to start a new one. A completed run loads its saved best model instead of training again.

For an interrupted run, rerun the preceding cells and select the previous run number here. The dataset will be rebuilt locally and checked against the saved fingerprint before resuming. Keep the same original dataset files in Drive. Checkpoints can only recover epochs that were actually saved.


In [ ]:
import torch, json, sys, platform, importlib.metadata
from datetime import datetime, timezone

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun this cell.')
OUTPUT_ROOT = PROJECT_DIR/'outputs'
OUTPUT_ROOT.mkdir(exist_ok=True)
existing = sorted([p for p in OUTPUT_ROOT.iterdir() if p.is_dir() and
                   ((p/'training'/'weights'/'last.pt').exists() or (p/'training_complete.json').exists())],reverse=True)
selection = ''
if existing:
    print('Saved runs:')
    for i,p in enumerate(existing,1): print(i,p.name)
    selection = input('Press Enter for a NEW run, or type the number of a run to resume: ').strip()
if selection:
    if not selection.isdigit() or not 1 <= int(selection) <= len(existing):
        raise ValueError('Enter one of the listed run numbers, or press Enter for a new run.')
    SAVE_DIR = existing[int(selection)-1]
    RESUME = True
else:
    run_name = 'edgevision_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    SAVE_DIR = OUTPUT_ROOT/run_name
    SAVE_DIR.mkdir()
    RESUME = False
LOCAL_DIR = Path('/content/helmet_edgevision')/SAVE_DIR.name
LOCAL_DIR.mkdir(parents=True,exist_ok=True)
DEVICE = 0
VERSIONS = {p:importlib.metadata.version(p) for p in ['ultralytics','torch','numpy','pillow','scipy']}
VERSIONS.update(python=sys.version,gpu=torch.cuda.get_device_name(0))
(SAVE_DIR/'environment_current.json').write_text(json.dumps(VERSIONS,indent=2))
if not RESUME: (SAVE_DIR/'environment.json').write_text(json.dumps(VERSIONS,indent=2))
print('GPU:', VERSIONS['gpu'])
print('Persistent results:', SAVE_DIR)
print('Resume:', RESUME)


## 4 — Name the main model and its training settings
This preserves **YOLOv8m**, the architecture in your original main notebook. Starting from pretrained `yolov8m.pt` is transfer learning, not training a detector from random weights. It helps when the custom dataset is relatively small.

The original notebook trained for 20 epochs. This version allows up to 100 with early stopping after 20 epochs without validation improvement. `best.pt` is chosen using validation fitness, not simply the final epoch. More epochs do not guarantee a better model.


In [ ]:
from ultralytics import YOLO

MAIN_MODEL = 'yolov8m.pt'
SETTINGS = {
    'seed':42, 'epochs':100, 'patience':20, 'imgsz':640, 'batch':8,
    'balance_training':True, 'capture_block_size':50,
    'confidence':0.25, 'expected_source_images':2392,
}
settings_path=SAVE_DIR/'settings.json'
if RESUME:
    SETTINGS=json.loads(settings_path.read_text())
else:
    settings_path.write_text(json.dumps(SETTINGS,indent=2))
print('Main model:',MAIN_MODEL)
print('Settings:',SETTINGS)


## 5 — Define file loading from the same Drive folder
The loader finds EdgeVision image names and YOLO labels inside ZIPs or extracted folders. It validates the class order and requires a matching image for every label. It copies the files into this run's local Colab folder and leaves the Drive originals unchanged.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, io, json, math, random, re, shutil, zipfile
import numpy as np
from PIL import Image, ImageDraw
from scipy.fft import dctn
import yaml

CLASS_NAMES = {0: 'accept-Helmet-', 1: 'non-Helmet-'}
SOURCE_CLASSES = ['BikeWithRider', 'NoHelmet', 'Helmet']
SOURCE_TO_TARGET = {2: 0, 1: 1}
# These four examples were inspected visually in the supplied sample.
KNOWN_EXCLUSIONS = {
    'Image_01355': 'Visible no-helmet foreground rider has no head annotation.',
    'Image_01384': 'Whole motorcycles incorrectly labelled NoHelmet.',
    'Image_01441': 'Two visible no-helmet foreground heads are unlabelled.',
    'Image_01512': 'Visible right-hand helmeted rider has no helmet annotation.',
}
IMAGE_PATTERN = re.compile(r'^Image_\d+\.(jpg|jpeg|png)$', re.I)
LABEL_PATTERN = re.compile(r'^Image_\d+\.txt$', re.I)

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def source_kind(filename):
    p = Path(filename.replace('\\','/')); name = p.name
    if IMAGE_PATTERN.match(name): return 'images', name
    if LABEL_PATTERN.match(name) and 'yolo' in [s.lower() for s in p.parts]: return 'labels', name
    if name.lower() == 'classes.txt' and 'yolo' in [s.lower() for s in p.parts]: return 'labels', 'classes.txt'
    if name.lower() in ['readme.txt','license.txt']: return 'metadata', name.upper()
    if name.lower() == 'annotations.json' and 'coco' in [s.lower() for s in p.parts]: return 'metadata', 'annotations.json'
    return None

def stage_sources(project_dir, output):
    """Read ZIPs/extracted source folders beside the notebook; never modify them."""
    project_dir, output = Path(project_dir), Path(output)
    if output.exists(): shutil.rmtree(output)  # This is the run's disposable staging folder only.
    output.mkdir(parents=True)
    copied = 0
    def put(stream, kind, name):
        nonlocal copied
        dest = output/kind/name; dest.parent.mkdir(parents=True,exist_ok=True)
        temp = dest.with_suffix(dest.suffix+'.incoming')
        with temp.open('wb') as f: shutil.copyfileobj(stream,f,1024*1024)
        if dest.exists():
            if file_sha(dest) != file_sha(temp):
                temp.unlink()
                if kind=='metadata': return
                raise ValueError(f'Conflicting copies of {name}. Keep only one dataset version in Helmet_Project.')
            temp.unlink()
        else:
            temp.replace(dest); copied += 1
            if copied%500==0: print('Source files copied:',copied)
    for archive in sorted(project_dir.glob('*.zip')):
        print('Reading ZIP:',archive.name)
        with zipfile.ZipFile(archive) as z:
            eligible=[(item,source_kind(item.filename)) for item in z.infolist() if not item.is_dir()]
            eligible=[(item,kind) for item,kind in eligible if kind]
            required=sum(item.file_size for item,_ in eligible)
            if required > shutil.disk_usage(output).free:
                raise OSError('Colab has insufficient local disk space for this archive.')
            for item,(kind,name) in eligible:
                # Flatten to validated basenames: archive paths are never used for extraction.
                with z.open(item) as f: put(f,kind,name)
    # Also support an extracted EdgeVision folder if preferred over a ZIP.
    for folder in sorted(project_dir.iterdir()):
        if not folder.is_dir() or folder.name in ['outputs','.ipynb_checkpoints']: continue
        for p in sorted(folder.rglob('*')):
            if p.is_file():
                kind=source_kind(str(p.relative_to(folder)))
                if kind:
                    with p.open('rb') as f: put(f,*kind)
    class_file=output/'labels'/'classes.txt'
    if not class_file.exists():
        raise FileNotFoundError('No labels/yolo/classes.txt found. Put the FULL EdgeVision dataset ZIP beside this notebook in Drive/MyDrive/Helmet_Project.')
    names=[s.strip() for s in class_file.read_text(encoding='utf-8-sig').splitlines() if s.strip()]
    if names != SOURCE_CLASSES:
        raise ValueError(f'Unexpected class list: {names}. This notebook is for EdgeVision V1.')
    images={p.stem:p for p in (output/'images').glob('*')} if (output/'images').exists() else {}
    labels={p.stem:p for p in (output/'labels').glob('Image_*.txt')}
    missing=sorted(set(labels)-set(images)); orphan=sorted(set(images)-set(labels))
    if missing or orphan:
        raise ValueError(f'Incomplete dataset: {len(images)} images and {len(labels)} labels. Missing images: {missing[:5]}; missing labels: {orphan[:5]}. Upload the FULL 2,392-image ZIP, not the 209-image sample.')
    print(f'Staged {len(images)} images and {len(labels)} matching YOLO labels.')
    return images, labels


## 6 — Read the complete dataset
This replaces the original API/download cell. The expected count refers to **original source files**, before cleaning. A missing-image error usually means the label-only ZIP or the 209-image sample was uploaded instead of the complete dataset.


In [ ]:
SOURCE_IMAGES, SOURCE_LABELS = stage_sources(PROJECT_DIR,LOCAL_DIR/'source')
if len(SOURCE_IMAGES)!=SETTINGS['expected_source_images']:
    raise ValueError(f"Expected 2392 EdgeVision V1 images, found {len(SOURCE_IMAGES)}. Use the complete original dataset.")
print('Full dataset loaded. No API key was required.')


## 7 — Define label cleaning and the two-class conversion
| Source ID / name | Treatment | Output ID / name |
|---|---|---|
| 0 / BikeWithRider | Ignore its boxes | Not a target class |
| 1 / NoHelmet | Keep valid head boxes | 1 / non-Helmet- |
| 2 / Helmet | Keep valid helmet boxes | 0 / accept-Helmet- |

The images still show motorcycles after removing the bike-box class. Only the prediction classes are reduced to two. This is not a motorcycle-to-person association system.

The function rejects malformed labels and invalid zero-area head boxes, clips boxes that slightly cross the image boundary, and removes duplicate head annotations. It does not invent labels for heads that were never annotated.


In [ ]:
def normalize_head_boxes(text):
    """Drop class 0; clip valid head boxes; reject malformed head-label images."""
    rows=[]; notes=[]
    for number,line in enumerate(text.splitlines(),1):
        if not line.strip(): continue
        fields=line.split()
        try:
            values=[float(v) for v in fields]
            if len(values)!=5 or not all(math.isfinite(v) for v in values): raise ValueError()
            cls,x,y,w,h=values
            if cls not in [0,1,2]: raise ValueError()
        except ValueError:
            return None,[f'malformed line {number}']
        if cls==0: continue
        if w<=0 or h<=0:
            return None,[f'zero/negative-area head box on line {number}']
        x1,y1,x2,y2=x-w/2,y-h/2,x+w/2,y+h/2
        clipped=[max(0.,min(1.,v)) for v in [x1,y1,x2,y2]]
        a,b,c,d=clipped
        if c<=a or d<=b: return None,[f'head box outside image on line {number}']
        if max(abs(u-v) for u,v in zip(clipped,[x1,y1,x2,y2]))>1e-6:
            notes.append(f'clipped head box on line {number}')
        row=[SOURCE_TO_TARGET[int(cls)],(a+c)/2,(b+d)/2,c-a,d-b]
        if any(row[0]==r[0] and max(abs(x-y) for x,y in zip(row[1:],r[1:]))<1e-6 for r in rows):
            notes.append(f'duplicate head box removed on line {number}')
        else: rows.append(row)
    return rows,notes


## 8 — Define image cleaning
This validates image decoding and dimensions, excludes images needing confirmed corrections, and checks exact duplicates. Head boxes wider or taller than 60% of the image are quarantined for review; some may be valid close-ups. Conflicting labels on identical images are excluded for review. Images with no head labels are not automatically treated as valid negative examples.

Images are saved with their aspect ratio preserved and maximum side 1,600 pixels to reduce disk traffic. YOLO later resizes/letterboxes them to 640 for training. Source images and labels remain unchanged.

Known exclusions: `Image_01355`, `Image_01384`, `Image_01441`, `Image_01512`. These have confirmed missing or wrong target labels in the sample. The cleaning report records these and every structural exclusion/correction. This is **not** a full manual reannotation of all 2,392 images.


In [ ]:
def clean_dataset(images, labels, output, metadata=None):
    """Validate every image; preserve normalized geometry; retain original sources."""
    output=Path(output)
    if output.exists(): shutil.rmtree(output)
    (output/'images').mkdir(parents=True);(output/'labels').mkdir()
    records=[];events=[];dimensions={}
    if metadata and Path(metadata).exists():
        meta=json.loads(Path(metadata).read_text())
        dimensions={Path(im['file_name']).stem:(im['width'],im['height']) for im in meta['images']}
    for i,(stem,path) in enumerate(sorted(images.items()),1):
        if stem in KNOWN_EXCLUSIONS:
            events.append({'image':stem,'action':'excluded','reason':KNOWN_EXCLUSIONS[stem]}); continue
        rows,notes=normalize_head_boxes(labels[stem].read_text(encoding='utf-8-sig'))
        if rows is None or not rows:
            reason='; '.join(notes) if rows is None else 'No target head labels; not assumed to be a true negative.'
            events.append({'image':stem,'action':'excluded','reason':reason});continue
        if any(r[3]>.6 or r[4]>.6 for r in rows):
            events.append({'image':stem,'action':'quarantined',
                           'reason':'Head box spans over 60% of image width or height; may be a whole-vehicle label or a valid close-up. Restore only after visual review.'})
            continue
        try:
            with Image.open(path) as opened:
                if opened.getexif().get(274,1) not in [0,1]:
                    raise ValueError('Nontrivial EXIF rotation requires visual review before box conversion.')
                im=opened.convert('RGB');im.load()
            if stem in dimensions and im.size!=dimensions[stem]:
                raise ValueError('Image dimensions differ from the annotation metadata.')
        except Exception as e:
            events.append({'image':stem,'action':'excluded','reason':str(e)});continue
        pixel_hash=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
        v=dctn(np.asarray(im.convert('L').resize((32,32)),dtype=float),type=2,norm='ortho')[:8,:8].ravel()[1:]
        phash=(v>np.median(v)).astype(int).tolist()
        # Limit file sizes while preserving aspect ratio and normalized boxes.
        # This never upscales images; Ultralytics later letterboxes to training size.
        im.thumbnail((1600,1600),Image.Resampling.LANCZOS)
        out_image=output/'images'/f'{stem}.jpg';im.save(out_image,quality=95)
        out_label=output/'labels'/f'{stem}.txt'
        text='\n'.join(str(int(r[0]))+' '+' '.join(f'{v:.8f}' for v in r[1:]) for r in rows)+'\n'
        out_label.write_text(text)
        records.append({'stem':stem,'image':str(out_image),'label':str(out_label),'boxes':rows,
                        'counts':[sum(r[0]==c for r in rows) for c in [0,1]],
                        'pixel_hash':pixel_hash,'phash':phash,'source_sha':file_sha(path),
                        'source_label_sha':file_sha(labels[stem])})
        for note in notes: events.append({'image':stem,'action':'corrected','reason':note})
        if i%400==0: print('Images checked:',i,'/',len(images))
    # Exact duplicate images with inconsistent annotations require review; do not choose a label arbitrarily.
    clusters=defaultdict(list)
    for r in records: clusters[r['pixel_hash']].append(r)
    remove=set()
    for rs in clusters.values():
        if len(rs)<2: continue
        signatures={tuple(sorted(tuple(round(v,6) for v in b) for b in r['boxes'])) for r in rs}
        excluded=rs[1:] if len(signatures)==1 else rs
        for r in excluded:
            remove.add(r['stem'])
            events.append({'image':r['stem'],'action':'excluded','reason':'Exact duplicate.' if len(signatures)==1 else 'Exact duplicate has conflicting labels; needs review.'})
    kept=[r for r in records if r['stem'] not in remove]
    for r in records:
        if r['stem'] in remove: Path(r['image']).unlink();Path(r['label']).unlink()
    if not kept: raise ValueError('No usable images remain.')
    return kept,events


## 9 — Run cleaning and save the audit
This takes several minutes for the full dataset. The report records what was removed or corrected. Counts distinguish source images, retained images and object boxes.


In [ ]:
import pandas as pd
from IPython.display import display

CLEAN_RECORDS, CLEAN_EVENTS = clean_dataset(
    SOURCE_IMAGES,SOURCE_LABELS,LOCAL_DIR/'clean',
    LOCAL_DIR/'source'/'metadata'/'annotations.json')
if len(CLEAN_RECORDS)<1000:
    raise RuntimeError(f'Only {len(CLEAN_RECORDS)} images survived. Inspect the input dataset and cleaning events before training.')
CLEAN_COUNTS=[sum(r['counts'][c] for r in CLEAN_RECORDS) for c in [0,1]]
print('Original images:',len(SOURCE_IMAGES))
print('Retained unique images:',len(CLEAN_RECORDS))
print('Retained boxes [helmet, no helmet]:',CLEAN_COUNTS)
print('Cleaning events:',len(CLEAN_EVENTS))
(SAVE_DIR/'cleaning_report.json').write_text(json.dumps(CLEAN_EVENTS,indent=2))
for p in (LOCAL_DIR/'source'/'metadata').glob('*.TXT'):
    shutil.copy2(p,SAVE_DIR/('SOURCE_'+p.name))
display(pd.DataFrame(CLEAN_EVENTS).head(25))


## 10 — Group related images before splitting
Near-identical photos must not appear in both training and evaluation. The code groups perceptually similar images and, conservatively, blocks of 50 consecutive source filenames. It then searches for an approximately **70% training / 15% validation / 15% test** allocation with both classes in each split.

These filename blocks are a precaution, **not verified capture-session identities**. Similar backgrounds or riders across distant filenames can still escape detection. A proper future benchmark should use separate cameras, locations or recording sessions.

Optional advanced input: a `capture_groups.csv` beside the notebook with columns `image,group` can supply known recording groups (for example `Image_00001.jpg,video_01`). This is not needed to run the notebook. Actual group assignments and potential duplicate pairs are saved for review.


In [ ]:
def assign_grouped_splits(records,seed=42,block_size=50,capture_groups=None):
    """Group visually similar photos plus adjacent filename blocks, then stratify."""
    n=len(records);parent=list(range(n))
    def root(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]];i=parent[i]
        return i
    def merge(i,j): parent[root(j)]=root(i)
    owner={};pairs=[]
    for i,r in enumerate(records):
        # Consecutive names are a conservative proxy, NOT verified video identities.
        number=int(re.search(r'(\d+)$',r['stem']).group(1))
        group=('block',(number-1)//block_size)
        if capture_groups and r['stem'] in capture_groups: group=('capture',capture_groups[r['stem']])
        if group in owner: merge(i,owner[group])
        else:owner[group]=i
    hashes=np.asarray([r['phash'] for r in records],dtype=np.uint8)
    for i in range(n-1):
        distances=np.count_nonzero(hashes[i+1:]!=hashes[i],axis=1)
        for offset in np.where(distances<=6)[0]:
            j=i+1+int(offset);merge(i,j)
            pairs.append([records[i]['stem'],records[j]['stem'],int(distances[offset])])
    groups=defaultdict(list)
    for i,r in enumerate(records): groups[root(i)].append(i)
    if len(groups)<3: raise ValueError('Too few independent image groups for train/validation/test splitting.')
    totals=np.sum([r['counts'] for r in records],axis=0)
    if min(totals)<30: raise ValueError('Too few head examples in one class.')
    fractions=np.array([.70,.15,.15]);rng=random.Random(seed);best=None
    for _ in range(800):
        keys=list(groups);rng.shuffle(keys);sizes=np.zeros(3,int);counts=np.zeros((3,2),int);assignment={}
        for gid in keys:
            idx=int(np.argmax(fractions*n-sizes));assignment[gid]=idx
            sizes[idx]+=len(groups[gid]);counts[idx]+=np.sum([records[i]['counts'] for i in groups[gid]],axis=0)
        if min(sizes)==0 or np.any(counts<10):continue
        score=float(np.sum(abs(sizes/n-fractions))+np.sum(abs(counts/totals-fractions[:,None])))
        if best is None or score<best[0]:best=(score,assignment)
    if best is None: raise ValueError('Cannot form three class-representative grouped splits. Review capture groups.')
    for gid,indices in groups.items():
        for i in indices:
            records[i]['group']=str(gid)
            records[i]['split']=['train','val','test'][best[1][gid]]
    for field in ['group','pixel_hash']:
        splits=defaultdict(set)
        for r in records:splits[r[field]].add(r['split'])
        assert all(len(s)==1 for s in splits.values()),f'Leakage in {field}'
    return records,pairs


In [ ]:
capture_groups=None
capture_file=PROJECT_DIR/'capture_groups.csv'
if capture_file.exists():
    with capture_file.open(newline='') as f:
        capture_groups={Path(r['image']).stem:r['group'] for r in csv.DictReader(f)}
GROUPED_RECORDS, NEAR_PAIRS = assign_grouped_splits(
    CLEAN_RECORDS,seed=SETTINGS['seed'],block_size=SETTINGS['capture_block_size'],
    capture_groups=capture_groups)
(SAVE_DIR/'similar_image_pairs.json').write_text(json.dumps(NEAR_PAIRS,indent=2))
print('Potential similar-image pairs grouped together:',len(NEAR_PAIRS))
print('Independent groups used:',len({r['group'] for r in GROUPED_RECORDS}))


## 11 — Prepare YOLO folders and balance training exposure
The original annotations contain 2,810 no-helmet and 1,672 helmet boxes. Images and boxes are different counts: one photo may contain both classes and several riders.

When needed, minority-class training images are repeated, with at most two additional copies per source image. This aims for similar object exposure during training; it does **not create new independent images**. Validation and test images are not repeated. Balancing supports learning both classes rather than guaranteeing any accuracy.


In [ ]:
def write_training_dataset(records,output,balance=True,seed=42):
    output=Path(output)
    if output.exists():shutil.rmtree(output)
    rows=[]
    for r in records:
        row={'stem':r['stem'],'group':r['group'],'split':r['split'],'counts':r['counts'],'repeat':0}
        for kind in ['image','label']:
            src=Path(r[kind]);dest=output/(kind+'s')/r['split']/src.name
            dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dest);row[kind]=str(dest)
        rows.append(row)
    if balance:
        train=[r for r in rows if r['split']=='train'];totals=np.sum([r['counts'] for r in train],axis=0)
        minor=int(np.argmin(totals));major=1-minor
        eligible=[r for r in train if r['counts'][minor]>r['counts'][major]]
        random.Random(seed).shuffle(eligible)
        for rep in [1,2]:
            for r in eligible:
                if totals[minor]>=.95*totals[major]:break
                row=dict(r,repeat=rep)
                for kind in ['image','label']:
                    src=Path(r[kind]);dest=src.with_stem(src.stem+f'_repeat{rep}')
                    shutil.copy2(src,dest);row[kind]=str(dest)
                rows.append(row);totals+=r['counts']
            if totals[minor]>=.95*totals[major]:break
    cfg={'path':str(output.resolve()),'train':'images/train','val':'images/val','test':'images/test','nc':2,'names':CLASS_NAMES}
    path=output/'data.yaml';path.write_text(yaml.safe_dump(cfg,sort_keys=False))
    summary={}
    for split in ['train','val','test']:
        unique=[r for r in rows if r['split']==split and r['repeat']==0];exposed=[r for r in rows if r['split']==split]
        summary[split]={'unique_images':len(unique),'image_exposures':len(exposed),
                        'head_boxes':[sum(r['counts'][c] for r in unique) for c in [0,1]],
                        'box_exposures':[sum(r['counts'][c] for r in exposed) for c in [0,1]]}
    return path,rows,summary

def dataset_fingerprint(records,settings):
    content=[{k:r[k] for k in ['stem','source_sha','source_label_sha','boxes','split','group']} for r in sorted(records,key=lambda r:r['stem'])]
    return hashlib.sha256(json.dumps({'data':content,'settings':settings},sort_keys=True).encode()).hexdigest()


In [ ]:
DATA_YAML, DATA_ROWS, DATA_SUMMARY = write_training_dataset(
    GROUPED_RECORDS,LOCAL_DIR/'dataset',balance=SETTINGS['balance_training'],seed=SETTINGS['seed'])
data_settings={k:SETTINGS[k] for k in ['seed','balance_training','capture_block_size']}
FINGERPRINT=dataset_fingerprint(GROUPED_RECORDS,data_settings)
fingerprint_file=SAVE_DIR/'dataset_fingerprint.txt'
if RESUME and fingerprint_file.exists() and fingerprint_file.read_text().strip()!=FINGERPRINT:
    raise RuntimeError('The dataset, cleaning, groups or split changed. Start a NEW run rather than resuming this checkpoint.')
fingerprint_file.write_text(FINGERPRINT)
(SAVE_DIR/'split_manifest.json').write_text(json.dumps([
    {k:r[k] for k in ['stem','source_sha','source_label_sha','pixel_hash','boxes','counts','split','group']}
    for r in GROUPED_RECORDS],indent=2))
(SAVE_DIR/'dataset_summary.json').write_text(json.dumps(DATA_SUMMARY,indent=2))
shutil.copy2(DATA_YAML,SAVE_DIR/'data.yaml')
display(pd.DataFrame(DATA_SUMMARY).T)
print('Box-count order: [helmet, no helmet].')
print('YOLO config:',DATA_YAML)


## 12 — Preview the cleaned training labels
Green = helmet; red = no helmet. Inspect the visible heads and boxes. A box should surround a helmet/head, not an entire motorcycle. Examples below come only from training. They are a spot check, not proof that all labels are correct.

If you notice incorrect/missing labels, correct the source annotations and start a new run; do not tune using the test images. Incorrect supervision can reduce no-helmet recall even when class counts look good.


In [ ]:
import matplotlib.pyplot as plt

sample=[r for r in DATA_ROWS if r['split']=='train' and r['repeat']==0]
random.Random(SETTINGS['seed']).shuffle(sample)
fig,axes=plt.subplots(4,4,figsize=(14,15))
for ax,row in zip(axes.flat,sample[:16]):
    im=Image.open(row['image']).convert('RGB');w,h=im.size;draw=ImageDraw.Draw(im)
    for line in Path(row['label']).read_text().splitlines():
        cls,x,y,bw,bh=map(float,line.split())
        draw.rectangle(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h),
                       outline='lime' if cls==0 else 'red',width=max(2,w//300))
    ax.imshow(im);ax.set_title(row['stem']);ax.axis('off')
plt.tight_layout()
fig.savefig(SAVE_DIR/'cleaned_training_preview.jpg',dpi=160)
plt.show()


## 13 — Configure augmentation in a separate cell
**Augmentation still happens automatically with Ultralytics when the dataset comes from files.** An API is only a download method; it has nothing to do with whether augmentation runs.

This cell explicitly configures colour changes, small rotations, translation, scale, horizontal flips and mosaic. Bounding boxes transform with the images. Vertical flips are disabled. Mosaic stops during the final 10 planned epochs. Validation/test do not receive random training augmentation.

The training loader generates augmented samples in memory. It does not permanently multiply your dataset into an augmentation folder. Saved batch-preview images and `args.yaml` provide evidence of augmentation for your viva.


In [ ]:
AUGMENTATION = {
    'hsv_h':0.015, 'hsv_s':0.4, 'hsv_v':0.3,
    'degrees':5.0, 'translate':0.1, 'scale':0.3,
    'shear':0.0, 'perspective':0.0,
    'flipud':0.0, 'fliplr':0.5,
    'mosaic':0.5, 'mixup':0.0, 'copy_paste':0.0,
    'close_mosaic':10,
}
aug_file=SAVE_DIR/'augmentation.json'
if RESUME:
    AUGMENTATION=json.loads(aug_file.read_text())
else:
    aug_file.write_text(json.dumps(AUGMENTATION,indent=2))
print(json.dumps(AUGMENTATION,indent=2))


## 14 — Train or resume the main model
This is the long-running step. Checkpoints are written to Drive under this run's `training/weights` folder. Keep Colab connected; no duration is guaranteed. `best.pt` is the checkpoint with the best validation fitness from this run; it is not automatically the best possible model for all real-world situations.

`last.pt` saves training progress. Early stopping can finish before epoch 100. If Colab reports GPU memory exhaustion, start a new run with a smaller batch value (4) in cell 4; ask for help if you prefer not to edit it. Resuming uses the saved training state and parameters.


In [ ]:
TRAIN_DIR=SAVE_DIR/'training'
LAST_PT=TRAIN_DIR/'weights'/'last.pt'
BEST_PT=TRAIN_DIR/'weights'/'best.pt'
completed=(SAVE_DIR/'training_complete.json').exists()
if completed:
    assert BEST_PT.exists(), 'Saved run is marked complete but its best.pt is missing.'
    print('Training already completed. Loading its saved best checkpoint.')
elif RESUME:
    if not LAST_PT.exists():
        raise FileNotFoundError('No saved last.pt is available for this run. Start a new run.')
    model=YOLO(str(LAST_PT))
    if model.ckpt.get('epoch',-1)<0 and BEST_PT.exists():
        print('Saved training finished before the completion marker was written; using best.pt.')
    else:
        model.train(resume=True,device=DEVICE)
else:
    if TRAIN_DIR.exists():
        raise RuntimeError('This run already has a training folder. Rerun cell 3 to resume it or start a new run; do not overwrite its checkpoints.')
    model=YOLO(MAIN_MODEL)
    model.train(
        data=str(DATA_YAML),epochs=SETTINGS['epochs'],patience=SETTINGS['patience'],
        imgsz=SETTINGS['imgsz'],batch=SETTINGS['batch'],device=DEVICE,
        workers=2,seed=SETTINGS['seed'],deterministic=True,
        project=str(SAVE_DIR),name='training',exist_ok=False,
        optimizer='AdamW',lr0=0.001,lrf=0.01,weight_decay=0.0005,
        pretrained=True,single_cls=False,cache=False,amp=True,
        plots=True,save=True,save_period=10,**AUGMENTATION,
    )
assert BEST_PT.is_file(), 'Training did not produce the expected best.pt.'
best_model=YOLO(str(BEST_PT))
assert {int(k):v for k,v in best_model.names.items()}==CLASS_NAMES, best_model.names
(SAVE_DIR/'training_complete.json').write_text(json.dumps({'best':str(BEST_PT),'model':MAIN_MODEL},indent=2))
shutil.copy2(BEST_PT,SAVE_DIR/'best.pt')
print('New model saved persistently:',SAVE_DIR/'best.pt')


## 15 — Display saved augmentation examples and training curves
For your viva: **"Ultralytics augments batches during training. These `train_batch*.jpg` files show examples, and `args.yaml` records the settings. We do not save every dynamically augmented image."**

These are actual training batch previews, not illustrations generated separately. The confusion matrix and prediction previews are additional evidence; they must be interpreted with the relevant split and class-specific metrics.


In [ ]:
from IPython.display import Image as NotebookImage

for p in sorted(TRAIN_DIR.glob('train_batch*.jpg'))[:3]:
    print(p.name);display(NotebookImage(filename=str(p),width=1000))
for name in ['results.png','confusion_matrix.png']:
    p=TRAIN_DIR/name
    if p.exists():display(NotebookImage(filename=str(p),width=1000))
print('Exact settings used:',TRAIN_DIR/'args.yaml')


## 16 — Define evaluation at your app's threshold
Detection should be evaluated with precision, recall and mAP, not an undefined single "accuracy" number. No-helmet **recall** shows how many labelled no-helmet heads are found. Precision shows how many no-helmet predictions are correct.

Ultralytics' summary precision/recall can correspond to its selected confidence point. The helper below separately evaluates at **confidence 0.25**, matching the original app's usual default, with one-to-one class-aware matching at IoU 0.5 and NMS IoU 0.7. This exposes missed heads at the threshold you actually use.


In [ ]:
def intersection_over_union(a,b):
    ix = max(0., min(a[2],b[2])-max(a[0],b[0]))
    iy = max(0., min(a[3],b[3])-max(a[1],b[1]))
    intersection = ix*iy
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - intersection
    return intersection/union if union>0 else 0.

def match_detections(predictions, ground_truth, threshold=.5):
    # predictions: [class, confidence, x1,y1,x2,y2]; truth: [class,x1,y1,x2,y2]
    tp, fp, fn = [0,0], [0,0], [0,0]
    used = set()
    for cls, confidence, *box in sorted(predictions, key=lambda p:-p[1]):
        cls = int(cls)
        candidates = [(intersection_over_union(box,gt[1:]), j)
                      for j,gt in enumerate(ground_truth) if int(gt[0])==cls and j not in used]
        score, j = max(candidates, default=(0.,-1))
        if score >= threshold:
            tp[cls]+=1; used.add(j)
        else:
            fp[cls]+=1
    for j,gt in enumerate(ground_truth):
        if j not in used:fn[int(gt[0])]+=1
    return tp,fp,fn

def wilson_interval(success,total,z=1.96):
    if total == 0:return [None,None]
    p=success/total;den=1+z*z/total
    centre=(p+z*z/(2*total))/den
    half=z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return [max(0.,centre-half),min(1.,centre+half)]

def evaluate_at_app_threshold(model, rows, split, device, confidence=.25):
    counts = [[0,0],[0,0],[0,0]]
    for row in rows:
        if row['split']!=split or row['repeat']!=0:continue
        truth = []
        for line in Path(row['label']).read_text().splitlines():
            cls,x,y,w,h=map(float,line.split())
            truth.append([int(cls),x-w/2,y-h/2,x+w/2,y+h/2])
        result = model.predict(source=row['image'],conf=confidence,iou=.7,
                               imgsz=640,device=device,verbose=False)[0]
        predictions = []
        if result.boxes is not None:
            predictions = [[int(cls),float(conf),*box] for cls,conf,box in zip(
                result.boxes.cls.cpu().tolist(),result.boxes.conf.cpu().tolist(),
                result.boxes.xyxyn.cpu().tolist())]
        matched=match_detections(predictions,truth)
        for k in range(3):
            for c in range(2):counts[k][c]+=matched[k][c]
    report={}
    for c,name in CLASS_NAMES.items():
        tp,fp,fn = [v[c] for v in counts]
        p=tp/(tp+fp) if tp+fp else 0.
        r=tp/(tp+fn) if tp+fn else 0.
        report[name]={'tp':tp,'fp':fp,'fn':fn,'precision':p,'recall':r,
                      'f1':2*p*r/(p+r) if p+r else 0.,
                      'recall_95_percent_interval':wilson_interval(tp,tp+fn)}
    return {'confidence':confidence,'matching_iou':.5,'nms_iou':.7,'classes':report}


## 17 — Evaluate the held-out test split
The test split is evaluated after checkpoint selection. It must not be used to choose the training recipe or confidence threshold. For further tuning, use validation, then evaluate on a fresh independent test set when possible.

The report includes both classes, mAP50, mAP50–95 and no-helmet precision/recall at confidence 0.25. The recall interval is a rough binomial interval; related images/heads violate independence, so it can understate real-world uncertainty.


In [ ]:
metrics=best_model.val(
    data=str(DATA_YAML),split='test',imgsz=SETTINGS['imgsz'],batch=SETTINGS['batch'],
    device=DEVICE,conf=0.001,plots=True,project=str(SAVE_DIR/'evaluation'),name='test',exist_ok=True)
operating=evaluate_at_app_threshold(best_model,DATA_ROWS,'test',DEVICE,SETTINGS['confidence'])
REPORT={
    'model':MAIN_MODEL,'classes':CLASS_NAMES,'source_images':len(SOURCE_IMAGES),
    'retained_unique_images':len(GROUPED_RECORDS),'split_summary':DATA_SUMMARY,
    'dataset_fingerprint':FINGERPRINT,'test_map50':float(metrics.box.map50),
    'test_map50_95':float(metrics.box.map),
    'class_ap':{CLASS_NAMES[int(c)]:{'ap50':float(a50),'ap50_95':float(a)}
                for c,a50,a in zip(metrics.box.ap_class_index,metrics.box.ap50,metrics.box.ap)},
    'at_app_threshold':operating,
    'limitations':['Only 209 source images were visually reviewed before notebook construction.',
                   'Heuristic grouping cannot guarantee all related captures stay together.',
                   'Webcam generalization and motorcycle/head association are not established.'],
}
(SAVE_DIR/'evaluation.json').write_text(json.dumps(REPORT,indent=2))
display(pd.DataFrame(operating['classes']).T)
print('Test mAP50:',round(metrics.box.map50,4))
print('Test mAP50–95:',round(metrics.box.map,4))
nh=operating['classes'][CLASS_NAMES[1]]
print('No-helmet precision:',round(nh['precision'],4),'recall:',round(nh['recall'],4))
print('75% targets met for no-helmet precision AND recall:',nh['precision']>=.75 and nh['recall']>=.75)
print('These targets do not prove that the model is better than the old one.')


## 18 — Optional comparison with your original checkpoint
If you put your trusted original two-class checkpoint in `Helmet_Project/best_original.pt`, this cell compares it on the same test images and prediction threshold. Otherwise it simply skips. Do not put the 13-class backup here.

Your old notebook's reported metrics used a different dataset, so comparing those numbers directly with the new test report would not establish improvement. Also check your actual camera images before replacing the app model.


In [ ]:
BASELINE=PROJECT_DIR/'best_original.pt'
if BASELINE.is_file():
    old_model=YOLO(str(BASELINE))
    if {int(k):v for k,v in old_model.names.items()}!=CLASS_NAMES:
        raise ValueError('The optional baseline does not have the original two-class mapping.')
    old_report=evaluate_at_app_threshold(old_model,DATA_ROWS,'test',DEVICE,SETTINGS['confidence'])
    (SAVE_DIR/'baseline_evaluation.json').write_text(json.dumps(old_report,indent=2))
    comparison=[]
    for label,report in [('original',old_report),('new',operating)]:
        for cls,values in report['classes'].items():
            comparison.append({'model':label,'class':cls,**values})
    display(pd.DataFrame(comparison))
else:
    print('Optional baseline skipped: best_original.pt was not provided.')


## 19 — Download best.pt and the evaluation report
Both files are already saved in Google Drive. Download `best.pt` for your existing app. If the browser blocks a download, use Drive and open the printed output folder instead.


In [ ]:
from google.colab import files
print('Saved model:',SAVE_DIR/'best.pt')
files.download(str(SAVE_DIR/'best.pt'))


In [ ]:
files.download(str(SAVE_DIR/'evaluation.json'))


## 20 — Use the result in your existing app
1. Back up the current app's `best.pt` as `best_original.pt`.
2. Put the newly downloaded file, named **`best.pt`**, beside **`app.py`**. Start the app from that folder as before.
3. Your app's first branch will load the two-class model. Keep `yolov11nbest.pt` for the existing fallback; no change to it is needed.
4. Use `ultralytics==8.3.162` for the initial app compatibility check, matching this notebook. Retain the app's other requirements.
5. Compare both models on the same independent motorcycle photos and your own camera captures. Keep the model that improves no-helmet detection without unacceptable false alarms. A higher training score alone is not enough.

The provided app uses `model(image)` and loads `best.pt` whenever the file exists. It does not switch to the 13-class model merely because an image produces no detections. **No detections means no confident detection, not proof of helmet wearing.**

The uploaded app currently provides image input; this notebook updates training and weights, not the camera interface. Your road-image training set also does not guarantee indoor webcam performance. Test that separately.

## Troubleshooting
- **"Create My Drive/Helmet_Project":** create that exact folder and place the notebook/dataset there.
- **"Incomplete dataset" or "Expected 2392":** use the full EdgeVision ZIP; the label-only ZIP and 209-image sample cannot train the full model.
- **Different classes:** a different dataset/version is present. Keep only the selected EdgeVision V1 source files.
- **CUDA out of memory:** start a new run with batch 4 instead of 8. Do not reduce annotation quality to make training fit.
- **Disconnected Colab:** reconnect, rerun cells, and choose the saved run number in cell 3. `last.pt` restores the saved training state after the dataset has been rebuilt and checked.
- **Invalid or suspicious labels:** inspect `cleaning_report.json`, the preview, and the originals. Missing labels require visual correction or exclusion; augmentation cannot fix them.
- **Model performs poorly:** use validation errors to guide better labels/diverse data; obtain a fresh test set before making a final claim. Do not repeatedly tune against the test report.

## Attribution and technical references
Dataset: Gajjar, Kandarp; Patel, Nancy; Patel, Ridham; Patel, Krutika; Dabhi, Chrisha; Trivedi, Himani; Goyani, Mahesh; Bhatt, Chintan (2026), *EdgeVision Dataset*, Mendeley Data V1, DOI **10.17632/j82bnw7gsr.1**. The supplied licence states **CC BY 4.0**. Retain its README/licence and acknowledge the conversion, cleaning, resizing and splitting performed here.

- [Ultralytics training and checkpoint documentation](https://docs.ultralytics.com/modes/train/)
- [Ultralytics augmentation documentation](https://docs.ultralytics.com/guides/yolo-data-augmentation/)
- [Ultralytics validation documentation](https://docs.ultralytics.com/modes/val/)

This is a revised training workflow based on your original `HELMETY8_v4.ipynb`. It preserves the original main architecture/class interface and replaces its online data cell, fixed 20-epoch training command and hard-coded result paths. Installed versions, settings, split assignments and reports are saved per run. Numerical results can still vary with hardware/software.


In [ ]:
for p in (LOCAL_DIR/'source'/'metadata').glob('*.TXT'):
    shutil.copy2(p,SAVE_DIR/('SOURCE_'+p.name))
print('Finished. Model and reports remain in:',SAVE_DIR)
